# 08 · RL with verifiable rewards and GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/08_grpo_rlvr.ipynb) &nbsp; 📖 Concept deck: [RLVR and GRPO](https://nilesh-salpe.github.io/visual-ai/post-training/08-rlvr-grpo/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** a verifiable reward, group-relative advantages (GRPO), and a critic-free RL loop that teaches a toy "model" to do arithmetic, plus a look at why groups where every answer is right or wrong give no signal.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

QUESTIONS = [(a, b) for a in range(10) for b in range(10)]      # "what is a + b?"
N_ANS = 19                                                       # answers 0..18
def base_logits(seed=0):
    """A weak 'base model': some mass on the right answer, more on near misses and a few favourite numbers."""
    rng = np.random.default_rng(seed); L = rng.normal(0, 1.0, (len(QUESTIONS), N_ANS))
    for q, (a, b) in enumerate(QUESTIONS):
        L[q, a + b] += 1.5; L[q, max(0, a + b - 1)] += 1.0; L[q, min(18, a + b + 1)] += 1.0
    return L
THETA0 = base_logits()
def accuracy(theta):
    p = softmax(theta); return float(np.mean([p[q, a + b] for q, (a, b) in enumerate(QUESTIONS)]))
print(f"base model: expected accuracy {accuracy(THETA0):.2f}")

## Task 1: A verifiable reward

For maths, code and logic you can often **check** an answer instead of asking a reward model. DeepSeek-R1-Zero uses rule-based rewards: an *accuracy* reward (is the final answer correct?) and a *format* reward (is the reasoning inside the required tags?). Tülu 3 named this approach **RLVR** (RL with verifiable rewards).
Return 1.0 if `answer == a + b`, else 0.0. (No format term in this toy.)

In [ ]:
def verify(question, answer):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_verify(f):
    assert f((3, 4), 7) == 1.0 and f((3, 4), 8) == 0.0 and f((0, 0), 0) == 1.0

check("verify", _test_verify, verify)

## Task 2: Group-relative advantages

GRPO (DeepSeekMath, 2024) samples a **group** of G outputs per question and uses the group itself as the baseline:
$$\hat A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r)}$$
Every token of output *i* gets this advantage. No value network (critic) is needed: *"GRPO foregoes the critic model, instead estimating the baseline from group scores"*.
Return the advantages for one group. If all rewards are equal (std = 0), return zeros: the group carries no learning signal.

In [ ]:
def group_advantages(r, eps=1e-8):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_group_advantages(f):
    a = f(np.array([1.0, 0.0, 0.0, 1.0]))
    assert np.allclose(a, [1, -1, -1, 1]), f"got {a}"
    assert np.allclose(f(np.ones(5)), 0) and np.allclose(f(np.zeros(5)), 0), "all-equal groups give zero advantage"
    b = f(np.array([1.0, 0, 0, 0]))
    assert np.isclose(b.mean(), 0) and np.isclose(b.std(), 1)

check("group_advantages", _test_group_advantages, group_advantages)

## Task 3: A critic-free RL loop

Each iteration:
1. Pick `n_q` random questions. For each, sample **G** answers from softmax(θ[q]).
2. Score them with `verify` and compute group advantages.
3. Policy-gradient step: $\theta[q] \mathrel{+}= \frac{lr}{G}\sum_i \hat A_i\,(e_{o_i} - \pi_q)$, then subtract $\beta(\pi_q - \pi_{ref,q})$: a simple pull back toward the starting policy that stands in for GRPO's KL penalty (DeepSeek-R1 uses a KL coefficient of 0.001; we use a small β).

(Full GRPO also uses PPO's clipped ratio for several updates per batch; with one on-policy step the ratio is 1, so it drops out.) Return θ and the accuracy history.

In [ ]:
def train_grpo(G=8, iters=400, n_q=32, lr=1.0, beta=0.01, seed=0):
    rng = np.random.default_rng(seed); theta = THETA0.copy(); pref = softmax(THETA0); hist = []
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_train_grpo(f):
    th, h = f()
    assert h[0] < 0.4 and h[-1] > 0.9, f"accuracy should go from ~{accuracy(THETA0):.2f} to > 0.9, got {h[-1]:.2f}"

check("train_grpo", _test_train_grpo, train_grpo)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: A verifiable reward
def verify_solution(question, answer):
    return float(answer == question[0] + question[1])

check("verify (reference)", _test_verify, verify_solution)

In [ ]:
# @title Solution for Task 2: Group-relative advantages
def group_advantages_solution(r, eps=1e-8):
    s = r.std()
    return np.zeros_like(r) if s < eps else (r - r.mean()) / s

check("group_advantages (reference)", _test_group_advantages, group_advantages_solution)

In [ ]:
# @title Solution for Task 3: A critic-free RL loop
def train_grpo_solution(G=8, iters=400, n_q=32, lr=1.0, beta=0.01, seed=0):
    rng = np.random.default_rng(seed); theta = THETA0.copy(); pref = softmax(THETA0); hist = []
    for _ in range(iters):
        for q in rng.choice(len(QUESTIONS), n_q, replace=False):
            p = softmax(theta[q]); outs = rng.choice(N_ANS, G, p=p)
            A = group_advantages_solution(np.array([verify_solution(QUESTIONS[q], o) for o in outs]))
            grad = (np.bincount(outs, A, N_ANS) - A.sum() * p) / G
            theta[q] += lr * (grad - beta * (p - pref[q]))
        hist.append(accuracy(theta))
    return theta, hist

check("train_grpo (reference)", _test_train_grpo, train_grpo_solution)

---
## Experiments
**When does a group teach anything?** If all G answers are right, or all wrong, every advantage is 0. Questions the model always or never solves give no gradient, which is why RLVR data is filtered to problems of the right difficulty and why larger G helps on hard questions. Compute the fraction of "dead" groups for the base model at several G, then the effect of G on training.

In [ ]:
p_right = np.array([softmax(THETA0[q])[a + b] for q, (a, b) in enumerate(QUESTIONS)])
for G in [2, 4, 8, 16, 64]:
    dead = np.mean(p_right ** G + (1 - p_right) ** G)
    print(f"G={G:3d}: expected fraction of groups with zero signal = {dead:.2f}")
print()
for G in [2, 4, 16]:
    _, h = train_grpo_solution(G=G, iters=150)
    print(f"G={G:2d}: accuracy after 150 iterations {h[-1]:.2f}  (samples used: {150 * 32 * G:,})")

---
## Takeaways
* Where correctness can be checked (maths, code, formats), a rule-based reward replaces the learned reward model, and there is far less to hack.
* GRPO drops PPO's critic: the mean and std of a group of samples for the same prompt act as the baseline. That saves a whole model's worth of memory.
* DeepSeekMath: GRPO took MATH from 46.8% to 51.7%. DeepSeek-R1-Zero: RL alone (no SFT) took AIME 2024 pass@1 from 15.6% to 77.9%, with longer reasoning and self-reflection emerging along the way.